# 第十章：细胞组成分析

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

前面的章节中，我们完成了差异表达、富集分析、通讯分析和转录因子网络——这些都在问"同一种细胞在疾病中表达了什么不同的基因？"

但还有一个更基础的问题：疾病改变了不同细胞类型的比例吗？

这个问题乍看简单——数一数就行了嘛。但真正做起来，统计学和生物学上都有不少坑。今天我们来系统解决它。

## 为什么组成分析比你想象的更难

### 问题一：细胞计数 ≠ 组织中的真实比例

scRNA-seq 中某种细胞有多少个，取决于采样偏差：酶解效率、分选策略、细胞存活率。大而脆弱的肝细胞（hepatocytes）在消化过程中大量死亡，导致它们在 scRNA-seq 中的比例远低于组织中的真实比例。

所以，我们能分析的是"在这个实验条件下被捕获的细胞的组成"——不是"组织中的真实组成"。结论要谨慎。

### 问题二：比例数据是"组成性"的

这是最容易被忽视的统计陷阱。每个样本中所有细胞类型的比例加和为 1。如果 A 类型的比例上升了，那么其他类型的比例必然被"挤压"下降——即使它们的绝对数量没有变化。

这就是 compositionality problem：你以为看到了 B 类型的下降，但其实只是 A 类型增加的"镜像效应"。

### 问题三：样本量决定统计能力

我们的数据只有 5 healthy + 5 cirrhotic 供体。Mann-Whitney U 检验在每组 5 个样本时，即使效应量很大，也很难达到统计显著。这不代表没有差异——只是我们缺少检测差异的统计"能力"。

⚠️ 踩坑预警：每组 5 个供体的统计检验

当每组只有 5 个样本时，Mann-Whitney U 检验能达到的最小 p 值约为 0.008（两组完全不重叠时）。如果你测试 12 种细胞类型再做 FDR 校正，几乎不可能有显著结果。这是统计学限制，不是你分析出了问题。

Part A：描述性统计
——先看再检验

### 每种细胞类型的计数交叉表

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

adata = sc.read_h5ad("results/05_annotated.h5ad")

# 大类 × 条件交叉表
ct_counts = pd.crosstab(
    adata.obs["cell_type"],
    adata.obs["condition"],
    margins=True
)
print(ct_counts.to_string())
ct_counts.to_csv(
    "results/celltype_condition_counts.csv"
)

**输出：**

> 📊 输出：
> condition       Cirrhotic  Healthy    All
> cell_type
> B cells              1262      660   1922
> Endothelial          3985     3900   7885
> HSCs/Mesenchyme      1234     1026   2260
> Hepatocytes          1829     1201   3030
> Macrophages          1897     1752   3649
> Monocytes            2545     2079   4624
> NK cells             4576     6230  10806
> Plasma cells          418      243    661
> Proliferating         432      357    789
> T cells             10488     9921  20409
> cDC                  1195      937   2132
> pDC                   318      250    568
> All                 30179    28556  58735

两组的总细胞数接近（30,179 vs 28,556），这为比例分析提供了较好的基础。

### 堆叠条形图：按供体可视化

在做统计检验之前，我们需要看每个供体的组成——因为统计单位是供体，不是细胞。

In [ ]:
import matplotlib.pyplot as plt

# 计算每个供体中各细胞类型的比例
prop_df = (
    adata.obs
    .groupby(["donor", "condition", "cell_type"])
    .size()
    .unstack(fill_value=0)
)
prop_df = prop_df.div(prop_df.sum(axis=1), axis=0)
prop_df.to_csv(
    "results/celltype_donor_proportions.csv"
)

In [ ]:
# 堆叠条形图
fig, ax = plt.subplots(figsize=(12, 6))
prop_df.plot(kind="bar", stacked=True, ax=ax,
             colormap="tab20")
ax.set_ylabel("Proportion")
ax.set_title("Cell Type Composition per Donor")
ax.legend(bbox_to_anchor=(1.02, 1), fontsize=7)
plt.tight_layout()
plt.savefig(
    "results/figures/09_stacked_barplot.png",
    dpi=150, bbox_inches="tight"
)
plt.close()

图 1：细胞类型比例变化箱线图——Healthy vs Cirrhotic

图 1：每个供体的细胞类型组成堆叠条形图。 10 根柱子对应 10 个供体。观察要点：供体间差异有多大？Healthy 和 Cirrhotic 之间有肉眼可见的差异吗？

### Boxplot：逐细胞类型比较

In [ ]:
import seaborn as sns

# 长格式：每行 = 一个供体×一种细胞类型
prop_long = prop_df.reset_index().melt(
    id_vars=["donor", "condition"],
    var_name="cell_type",
    value_name="proportion"
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(
    data=prop_long, x="cell_type",
    y="proportion", hue="condition",
    ax=ax, palette={"Healthy": "#4C72B0",
                     "Cirrhotic": "#DD8452"}
)
ax.set_xticklabels(
    ax.get_xticklabels(), rotation=45, ha="right"
)
plt.tight_layout()
plt.savefig(
    "results/figures/09_boxplot_composition.png",
    dpi=150, bbox_inches="tight"
)
plt.close()

图 2：每个供体的细胞类型组成堆叠柱状图

图 2：各细胞类型在 Healthy vs Cirrhotic 中的比例 boxplot。 每个点代表一个供体。这是组成分析最重要的图——审稿人看到这张图就能判断你的分析是否可靠。

💡 Tip：为什么用 boxplot 而不是柱状图？

柱状图只显示均值，完全掩盖了供体间的变异。Boxplot 能同时展示中位数、四分位距和离群值——这对只有 5 个样本的数据尤为重要。审稿人第一个要看的就是"供体间变异有多大"。

## Part B：统计检验——Mann-Whitney U + FDR

### 为什么选 Mann-Whitney U

样本量太小（每组 n=5），不满足 t 检验的正态性假设
比例数据常有偏态分布
Mann-Whitney U（也叫 Wilcoxon rank-sum）是非参数检验，不需要分布假设

In [ ]:
from scipy.stats import mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests

results = []
cell_types = prop_long["cell_type"].unique()

for ct in cell_types:
    h = prop_long[
        (prop_long["cell_type"] == ct)
        & (prop_long["condition"] == "Healthy")
    ]["proportion"]
    c = prop_long[
        (prop_long["cell_type"] == ct)
        & (prop_long["condition"] == "Cirrhotic")
    ]["proportion"]

    stat, pval = mannwhitneyu(
        h, c, alternative="two-sided"
    )
    fc = c.median() / h.median() if h.median() > 0 \
        else np.inf

    results.append({
        "cell_type": ct,
        "median_Healthy": h.median(),
        "median_Cirrhotic": c.median(),
        "fold_change": fc,
        "MWU_pvalue": pval
    })

res_df = pd.DataFrame(results)

### FDR 校正：多重检验的"保险"

我们对 12 种细胞类型做了 12 次检验。如果不校正，在 α=0.05 水平上，有 ~46% 的概率至少出现一次假阳性。

In [ ]:
_, padj, _, _ = multipletests(
    res_df["MWU_pvalue"], method="fdr_bh"
)
res_df["padj"] = padj

# 按 fold change 排序
res_df = res_df.sort_values(
    "fold_change", ascending=False
)
print(res_df[["cell_type", "fold_change",
              "MWU_pvalue", "padj"]].to_string(
    index=False))

res_df.to_csv(
    "results/composition_test_results.csv",
    index=False
)

**输出：**

> 📊 输出：
>       cell_type  fold_change  MWU_pvalue   padj
>        B cells         2.88      0.0952  0.8900
>    Plasma cells        1.73      0.3095  0.8900
>     Hepatocytes        1.54      0.2222  0.8900
>            cDC         1.28      0.5476  0.8900
>       Monocytes        1.23      0.5476  0.8900
>    Macrophages         1.08      0.8413  0.8900
>   Proliferating        1.05      0.8413  0.8900
>  HSCs/Mesenchyme       1.02      0.8413  0.8900
>            pDC         1.01      1.0000  1.0000
>     Endothelial        0.97      0.8413  0.8900
>        T cells         0.93      0.5476  0.8900
>       NK cells         0.59      0.0952  0.8900

### 结果解读：没有统计显著 ≠ 没有差异

所有细胞类型的 FDR 校正后 p 值都 ≥ 0.89，没有任何一种达到统计显著。

但请注意两个趋势：

B cells FC=2.88：肝硬化中 B 细胞比例是健康的近 3 倍。虽然不显著，但效应量很大。
NK cells FC=0.59：肝硬化中 NK 细胞比例下降了约 40%。

为什么不显著？回到前面的警告——每组只有 5 个供体。Mann-Whitney U 检验在 n=5 vs n=5 时，即使两组完全不重叠，最小 p 值也只有 ~0.008。对 12 个检验做 FDR 校正后，几乎不可能有显著结果。

⚠️ 踩坑预警：生物学显著性 vs 统计显著性

统计检验回答的是"这个差异是否可能只是随机波动"。但在 n=5 的情况下，即使真实差异存在，我们也缺乏统计能力去检测它。在论文中，你可以说"B cells showed a trend of increase (FC=2.88, MWU p=0.095, not significant after FDR correction), which may reflect..."。报告效应量 + 置信区间比单纯看 p 值更有信息量。

### 补充：Chi-squared 检验

作为辅助，我们用 Chi-squared 检验测试总体组成是否有差异（忽略供体重复结构）：

In [ ]:
# 原始计数交叉表（去掉 margins）
ct_raw = pd.crosstab(
    adata.obs["cell_type"],
    adata.obs["condition"]
)
chi2, pval, dof, expected = chi2_contingency(ct_raw)
print(f"Chi-squared: {chi2:.1f}")
print(f"p-value: {pval:.2e}")
print(f"df: {dof}")

**输出：**

> 📊 输出：
> Chi-squared: 574.8
> p-value: 3.22e-116
> df: 11

Chi-squared 高度显著（p ≈ 3e-116）——说明 Healthy 和 Cirrhotic 的总体细胞组成确实不同。

但这个检验有一个严重缺陷：它把每个细胞当作独立观测，完全忽略了"细胞来自同一个供体"的嵌套结构。这会导致严重的假阳性膨胀（pseudoreplication）。所以 Chi-squared 的结果只能参考，不能作为主要统计依据。

💡 Tip：正确的统计单位是供体，不是细胞

在多供体 scRNA-seq 实验中，统计推断的"样本"是供体（biological replicate），不是单个细胞。来自同一供体的细胞不是独立的——它们共享了基因组、微环境和技术偏差。把细胞当"样本"做检验是 scRNA-seq 分析中最常见的统计错误之一。

## Part C：scCODA——贝叶斯组成分析（尝试与回退）

### 为什么需要专门的组成分析方法

标准统计检验（t-test、Mann-Whitney）不处理 compositionality。scCODA（Büttner et al., Nature Communications, 2021）是专门为此设计的贝叶斯方法：它建立了一个 Dirichlet-Multinomial 模型，能同时考虑所有细胞类型的比例变化，避免 compositionality 带来的假阳性。

### 尝试使用 pertpy

scCODA 已整合到 pertpy（Perturbation analysis toolkit）中：

In [ ]:
# 尝试加载 pertpy 中的 scCODA
try:
    import pertpy as pt
    sccoda = pt.tl.Sccoda()
    # 构建 compositional data
    sccoda_data = sccoda.load(
        adata,
        type="cell_level",
        generate_sample_level=True,
        cell_type_identifier="cell_type",
        sample_identifier="donor",
        covariate_obs=["condition"]
    )
    print("✅ pertpy scCODA loaded successfully")
except Exception as e:
    print(f"❌ pertpy import failed: {e}")
    print("   Falling back to MWU analysis")

**输出：**

> 📊 输出：
> ❌ pertpy import failed: cannot import name 'Sccoda'
>    from 'pertpy.tools' (version conflict)
>    Falling back to MWU analysis

⚠️ 踩坑预警：pertpy 版本兼容性

pertpy 的 API 在不同版本之间变化较大。pt.tl.Sccoda() 在 pertpy ≥0.7 中可能已改名或移除。如果你遇到类似问题，可以尝试：(1) 直接安装原始 scCODA 包 pip install sccoda；(2) 使用 pertpy 的固定版本 pip install pertpy==0.6.0；(3) 回退到 Mann-Whitney + 效应量报告。

在我们的案例中，即使 scCODA 成功运行，n=5 的样本量也可能导致贝叶斯估计的后验概率较宽——本质问题还是样本量不足。

## Part D：细分注释的组成分析

大类分析没发现统计显著差异，但细分亚群中可能隐藏着更有意义的变化。用同样的方法对 26 种 fine type 做分析：

In [ ]:
# 细分注释的供体比例
prop_fine = (
    adata.obs
    .groupby(["donor", "condition",
              "cell_type_fine"])
    .size()
    .unstack(fill_value=0)
)
prop_fine = prop_fine.div(
    prop_fine.sum(axis=1), axis=0
)

# 计算每种 fine type 的 median 差异
fine_diff = []
for ct in prop_fine.columns:
    h = prop_fine.loc[
        prop_fine.index.get_level_values(
            "condition") == "Healthy", ct
    ]
    c = prop_fine.loc[
        prop_fine.index.get_level_values(
            "condition") == "Cirrhotic", ct
    ]
    diff = c.median() - h.median()
    fine_diff.append({"fine_type": ct,
                      "median_diff": diff})

fine_df = pd.DataFrame(fine_diff).sort_values(
    "median_diff", ascending=False
)
print(fine_df.head(5).to_string(index=False))
print("...")
print(fine_df.tail(5).to_string(index=False))

**输出：**

> 📊 输出：
> Top 5 增加（Cirrhotic vs Healthy）:
>      fine_type  median_diff
>    Vascular EC      +0.1154
>    Hepatocytes      +0.0531
>      CD14 Mono      +0.0423
>        B cells      +0.0379
>            LAM      +0.0312
> 
> Top 5 减少（Cirrhotic vs Healthy）:
>      fine_type  median_diff
>    CD56dim NK      -0.0288
>      CD4 naive     -0.0301
>           cDC      -0.0344
>          MAIT      -0.0488
>  CD56bright NK     -0.0731

### 关键发现

几个值得关注的趋势（虽然统计不显著）：

Vascular EC ↑0.1154：血管内皮细胞在肝硬化中比例增加最多。这可能反映了肝硬化中新生血管形成和肝窦毛细血管化的过程。

LAM ↑0.0312：脂质相关巨噬细胞（即原文的 SAMac）比例增加——与 Ramachandran 2019 的核心发现一致。

CD56bright NK ↓0.0731：NK 细胞的减少主要来自 CD56bright 亚群。CD56bright NK 是肝脏驻留的主要 NK 亚群，它们的减少可能反映了肝硬化中免疫微环境的改变。

MAIT ↓0.0488：MAIT 细胞在肝硬化中减少，这与已有文献报道一致（MAIT 细胞在慢性肝病中被激活耗竭）。

---

### 🔬 方法选择指南

🔬 方法选择指南：细胞组成分析方法对比
方法统计模型处理组成性最少样本量推荐场景描述性统计 ✓可视化（堆叠柱图/箱线图）否1+初步探索、结果展示Mann-Whitney U ✓非参数秩和检验否≥3/组少样本、简单两组比较scCODA贝叶斯Dirichlet-Multinomial✅ 是≥5/组（推荐）需要考虑组成效应时（推荐）propellerlogit变换 + 经验贝叶斯部分≥3/组快速、频率学框架MiloRKNN图差异丰度(DA)部分≥3/组不依赖预定义细胞类型、连续状态DA-seq多尺度KNN密度差异否2+无标签探索性分析
我们的选择：Mann-Whitney U检验（逐细胞类型）。理由：

① 样本量限制：Healthy 3人 vs Cirrhotic 2人，总共仅5个生物学重复。这对贝叶斯方法（scCODA）和部分频率方法来说样本太少；
② 透明性：MWU检验的逻辑清晰——每个细胞类型独立检验，P值含义直观；
③ 务实态度：我们在正文中尝试了scCODA但因版本兼容问题未成功。本教程选择展示可靠运行的方法。

组成性问题（compositional bias）是什么

单细胞数据中，所有细胞类型的比例之和 = 100%。如果某种细胞真正增加了，其他细胞的相对比例就会"被动下降"，即使它们的绝对数量没有变化。这就是"组成性偏差"。

scCODA通过联合建模所有细胞类型的比例来解决这个问题——它设定一个"参考细胞类型"（假设不变的），然后检验其他类型相对于参考的变化。这在统计上比独立检验每个类型更合理，但需要更多的样本量。

实践建议：如果你有≥10个样本/组，强烈建议使用scCODA或propeller。如果样本量<5，MWU结果只能作为"趋势提示"——务必注明统计效力不足。

---

## 📖 与原文比较

与 Ramachandran et al., 2019 对照：

原文的核心发现之一是：肝硬化中 Kupffer 细胞（肝脏驻留巨噬细胞）被耗竭，取而代之的是来自外周血的 单核细胞来源的巨噬细胞（包括 SAMac）增加。

我们的分析部分验证了这一发现：

LAM（SAMac） 在肝硬化中确实呈增加趋势（+0.0312）
CD14 Mono（可能是 SAMac 的前体）也呈增加趋势（+0.0423）

但我们没有检测到 Kupffer 细胞比例的明显下降——这可能与我们的注释策略有关（Kupffer 细胞和其他组织驻留巨噬细胞的 marker 重叠，分离困难）。

原文使用了更多的统计方法和功能验证（包括 FACS、免疫荧光等）来支持组成变化的结论。仅靠 scRNA-seq 的细胞计数，在样本量有限时，统计证据不够充分——这进一步说明了多层验证的重要性。

## 方法论反思

### compositionality：一个被严重低估的问题

假设肝硬化导致 B 细胞绝对数量增加了 3 倍，但其他细胞都没变。在比例数据中，B 细胞的比例会上升，同时所有其他细胞的比例都会被"挤压"下降——即使它们的绝对数量完全没变。

你可能因此得出"NK 细胞在肝硬化中减少"的结论，但实际上 NK 细胞的数量没变——只是 B 细胞的增加"稀释"了它们的比例。

解决方案：

scCODA 等组成数据专用方法：建模时考虑 compositionality
报告绝对数量 + 比例：两者一起看
用独立实验验证：FACS、免疫组化等

### 统计能力不足时怎么办

当 n=5 导致所有检验都不显著时：

报告效应量（fold change），不要只看 p 值
用可视化补充：boxplot 比 p 值更有信息量
引用文献支持趋势：如果你的趋势和已发表文献一致，说明方向可能是对的
承认局限性：诚实地写出 "limited by sample size"

## 小结

这一篇我们完成了：

✅ 描述性统计：交叉表、堆叠条形图、boxplot
✅ Mann-Whitney U 检验 + FDR 校正（12 种大类 → 全部 padj ≥ 0.89）
✅ Chi-squared 辅助检验（显著但有 pseudoreplication 问题）
✅ scCODA 尝试（pertpy 导入失败，回退到 MWU）
✅ 细分注释的组成分析（26 种 fine type）
✅ 输出文件：celltype_condition_counts.csv、celltype_donor_proportions.csv、composition_test_results.csv

关键数字：

大类组成检验：0/12 种细胞类型达到统计显著
最大效应量：B cells FC=2.88（增加）、NK cells FC=0.59（减少）
细分亚群最大变化：Vascular EC ↑0.1154、CD56bright NK ↓0.0731

核心教训： 统计不显著 ≠ 没有生物学差异。在样本量有限时，效应量和可视化比 p 值更重要。组成数据的分析需要注意 compositionality 问题。

下一篇，我们将用轨迹分析和拟时序分析追踪细胞的分化轨迹——比如单核细胞是如何一步步变成 SAMac 的。这是从"静态快照"走向"动态推断"的关键一步。